# NC-GAT: Noise-Consistency Regularized Inter-Lead Graph Attention Network

Notebook implementasi end-to-end untuk klasifikasi multi-label ECG 12-lead.

Cakupan notebook ini:

1. Konfigurasi environment dan path dataset
2. Harmonisasi label PTB-XL dan Chapman-Shaoxing ke skema SNOMED-CT bersama
3. Preprocessing sinyal (resample, normalisasi, padding)
4. Augmentasi noise dan simulasi missing-lead
5. Dataset dan pembagian data berbasis sumber
6. Arsitektur model (CNN baseline, Transformer baseline, NC-GAT)
7. Utilitas training dan evaluasi
8. Eksperimen A sampai E
9. Ringkasan hasil akhir

Notebook ini dirancang untuk dijalankan di Kaggle Notebook dengan GPU aktif.
Path dataset mengacu pada Kaggle Dataset yang sudah ditambahkan sebagai input:

- PTB-XL: khyeh0719/ptb-xl-dataset
- Chapman-Shaoxing: erarayamorenzomuten/chapmanshaoxing-12lead-ecg-database

In [32]:
# Jalankan sekali di awal sesi. Sebagian besar library sudah tersedia di image Kaggle default.
!pip install -q wfdb

In [33]:
import os
import ast
import json
import math
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import scipy.signal as sps
from scipy.stats import spearmanr
from scipy.spatial.distance import jensenshannon
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score
from sklearn.metrics.pairwise import cosine_similarity

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import wfdb

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


## Konfigurasi Path dan Parameter Global

Path di bawah ini adalah path yang sudah diverifikasi berhasil terbaca pada tahap eksplorasi data.
Jika struktur folder Kaggle Dataset berubah, sesuaikan konstanta ini saja, tidak perlu mengubah
kode di bagian lain notebook.

In [34]:
PTBXL_ROOT = "/kaggle/input/datasets/khyeh0719/ptb-xl-dataset/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1"
CHAPMAN_ROOT = "/kaggle/input/datasets/erarayamorenzomuten/chapmanshaoxing-12lead-ecg-database/WFDB_ChapmanShaoxing"

DX_MAPPING_URL = "https://raw.githubusercontent.com/physionetchallenges/evaluation-2021/main/dx_mapping_scored.csv"
DX_MAPPING_PATH = "/kaggle/working/dx_mapping_scored.csv"

TARGET_FS = 500
TARGET_SECONDS = 10
TARGET_LENGTH = TARGET_FS * TARGET_SECONDS

LEAD_NAMES = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
N_LEADS = len(LEAD_NAMES)

# Mapping SCP-ECG (PTB-XL) ke kode SNOMED-CT, sudah divalidasi terhadap dx_mapping_scored.csv.
# 5 dari 6 kelas cocok persis dengan angka referensi resmi, kelas NSR memiliki selisih
# sekitar 3 persen yang diterima sebagai batas toleransi (lihat catatan metodologi di laporan).
SCP_TO_SNOMED = {
    "SR": 426783006,
    "SARRH": 426783006,
    "SBRAD": 426177001,
    "STACH": 427084000,
    "AFIB": 164889003,
    "1AVB": 270492004,
    "CLBBB": 164909002,
}

# Daftar kelas final, urutan tetap dan dipakai konsisten di seluruh notebook.
CLASS_CODES = sorted(set(SCP_TO_SNOMED.values()))
CLASS_NAMES = {
    426783006: "NSR",
    426177001: "SB",
    427084000: "STach",
    164889003: "AF",
    270492004: "IAVB",
    164909002: "LBBB",
}
NUM_CLASSES = len(CLASS_CODES)
print("Kelas target:", [CLASS_NAMES[c] for c in CLASS_CODES])

Kelas target: ['AF', 'LBBB', 'IAVB', 'SB', 'NSR', 'STach']


## Tahap 1: Harmonisasi Label

PTB-XL menyimpan label secara native dalam skema SCP-ECG (kolom scp_codes, berbentuk dictionary
string berisi kode dan tingkat keyakinan). Chapman-Shaoxing menyimpan label secara native dalam
kode SNOMED-CT langsung pada field #Dx di header WFDB.

Kedua sumber diharmonisasikan ke satu skema SNOMED-CT yang sama, menggunakan mapping manual yang
sudah divalidasi terhadap berkas referensi resmi dx_mapping_scored.csv dari PhysioNet Challenge 2021.
Berkas ini hanya dipakai sebagai kamus dan validator label, bukan sebagai sumber sinyal baru.

In [35]:
def download_dx_mapping():
    if not os.path.exists(DX_MAPPING_PATH):
        os.system(f"wget -q {DX_MAPPING_URL} -O {DX_MAPPING_PATH}")
    if not os.path.exists(DX_MAPPING_PATH):
        raise FileNotFoundError(
            "Gagal mengunduh dx_mapping_scored.csv. Periksa apakah toggle Internet "
            "pada pengaturan notebook Kaggle sudah aktif."
        )
    return pd.read_csv(DX_MAPPING_PATH)


def validate_mapping_against_reference(counter, df_mapping, source_column):
    print(f"Validasi distribusi kelas terhadap kolom referensi: {source_column}")
    for code in CLASS_CODES:
        found = counter.get(code, 0)
        ref_row = df_mapping[df_mapping["SNOMEDCTCode"] == code]
        reference = int(ref_row[source_column].values[0]) if len(ref_row) else None
        status = "cocok" if reference is not None and found == reference else "periksa kembali"
        print(f"  {CLASS_NAMES[code]:6s} kode={code}  hasil={found:6d}  referensi={reference}  ({status})")

In [36]:
def map_scp_to_snomed(scp_codes_str):
    scp_dict = ast.literal_eval(scp_codes_str)
    codes = set()
    for scp_code in scp_dict.keys():
        if scp_code in SCP_TO_SNOMED:
            codes.add(SCP_TO_SNOMED[scp_code])
    return sorted(codes)


def multi_hot(codes):
    vec = np.zeros(NUM_CLASSES, dtype=np.float32)
    for c in codes:
        if c in CLASS_CODES:
            vec[CLASS_CODES.index(c)] = 1.0
    return vec


def build_ptbxl_metadata():
    df = pd.read_csv(os.path.join(PTBXL_ROOT, "ptbxl_database.csv"))
    df["snomed_codes"] = df["scp_codes"].apply(map_scp_to_snomed)
    df["label_vec"] = df["snomed_codes"].apply(multi_hot)
    df = df[df["snomed_codes"].apply(len) > 0].reset_index(drop=True)

    # filename_hr menunjuk ke rekaman 500 Hz, sesuai TARGET_FS notebook ini.
    df["record_path"] = df["filename_hr"].apply(lambda p: os.path.join(PTBXL_ROOT, p))
    df["source"] = "ptbxl"

    counter = Counter([c for codes in df["snomed_codes"] for c in codes])
    return df[["record_path", "source", "snomed_codes", "label_vec", "strat_fold", "patient_id"]], counter

In [37]:
def parse_chapman_header(hea_path):
    dx_codes = []
    with open(hea_path, "r") as f:
        for line in f:
            if line.strip().startswith("#Dx"):
                raw = line.strip().replace("#Dx:", "")
                dx_codes = [int(c.strip()) for c in raw.split(",") if c.strip().isdigit()]
                break
    return dx_codes


def build_chapman_metadata():
    all_files = [f for f in os.listdir(CHAPMAN_ROOT) if f.endswith(".hea")]
    records = []
    for fname in all_files:
        hea_path = os.path.join(CHAPMAN_ROOT, fname)
        dx_codes = parse_chapman_header(hea_path)
        relevant = sorted(set(dx_codes) & set(CLASS_CODES))
        if len(relevant) == 0:
            continue
        record_path = os.path.join(CHAPMAN_ROOT, fname.replace(".hea", ""))
        records.append({
            "record_path": record_path,
            "source": "chapman",
            "snomed_codes": relevant,
            "label_vec": multi_hot(relevant),
        })

    df = pd.DataFrame(records)
    counter = Counter([c for codes in df["snomed_codes"] for c in codes])
    return df, counter

In [38]:
df_mapping = download_dx_mapping()

df_ptbxl, counter_ptbxl = build_ptbxl_metadata()
print(f"PTB-XL: {len(df_ptbxl)} rekaman relevan dari total data")
validate_mapping_against_reference(counter_ptbxl, df_mapping, "PTB_XL")

print()
df_chapman, counter_chapman = build_chapman_metadata()
print(f"Chapman-Shaoxing: {len(df_chapman)} rekaman relevan dari total data")
validate_mapping_against_reference(counter_chapman, df_mapping, "Chapman_Shaoxing")

PTB-XL: 20635 rekaman relevan dari total data
Validasi distribusi kelas terhadap kolom referensi: PTB_XL
  AF     kode=164889003  hasil=  1514  referensi=1514  (cocok)
  LBBB   kode=164909002  hasil=   536  referensi=536  (cocok)
  IAVB   kode=270492004  hasil=   797  referensi=797  (cocok)
  SB     kode=426177001  hasil=   637  referensi=637  (cocok)
  NSR    kode=426783006  hasil= 17552  referensi=18092  (periksa kembali)
  STach  kode=427084000  hasil=   826  referensi=826  (cocok)

Chapman-Shaoxing: 9119 rekaman relevan dari total data
Validasi distribusi kelas terhadap kolom referensi: Chapman_Shaoxing
  AF     kode=164889003  hasil=  1780  referensi=1780  (cocok)
  LBBB   kode=164909002  hasil=   205  referensi=205  (cocok)
  IAVB   kode=270492004  hasil=   247  referensi=247  (cocok)
  SB     kode=426177001  hasil=  3889  referensi=3889  (cocok)
  NSR    kode=426783006  hasil=  1826  referensi=1826  (cocok)
  STach  kode=427084000  hasil=  1568  referensi=1568  (cocok)


## Tahap 2: Preprocessing Sinyal

Setiap rekaman dibaca melalui wfdb, diresample ke frekuensi sampling seragam, dinormalisasi
per lead, kemudian dipotong atau diberi padding agar seluruh rekaman memiliki panjang waktu
yang sama. Bentuk akhir setiap rekaman adalah tensor berukuran [12, TARGET_LENGTH].

In [39]:
def resample_signal(signal, original_fs, target_fs):
    if original_fs == target_fs:
        return signal
    n_samples_target = int(signal.shape[1] * target_fs / original_fs)
    return sps.resample(signal, n_samples_target, axis=1)


def normalize_signal(signal):
    mean = signal.mean(axis=1, keepdims=True)
    std = signal.std(axis=1, keepdims=True) + 1e-8
    return (signal - mean) / std


def pad_or_crop(signal, target_length):
    n_leads, length = signal.shape
    if length == target_length:
        return signal
    if length > target_length:
        start = (length - target_length) // 2
        return signal[:, start:start + target_length]
    pad_total = target_length - length
    pad_left = pad_total // 2
    pad_right = pad_total - pad_left
    return np.pad(signal, ((0, 0), (pad_left, pad_right)), mode="constant")


def load_and_preprocess(record_path):
    record = wfdb.rdrecord(record_path)
    signal = record.p_signal.T.astype(np.float32)
    signal = resample_signal(signal, record.fs, TARGET_FS)
    signal = pad_or_crop(signal, TARGET_LENGTH)
    signal = normalize_signal(signal)
    return signal.astype(np.float32)

## Tahap 3: Augmentasi Noise dan Missing-Lead

Fungsi berikut sudah diuji terpisah dengan data sintetis sebelum dipasang ke pipeline utama.
Noise ditambahkan pada level SNR tertentu, bukan amplitudo tetap, agar tingkat gangguan
dapat dikendalikan secara konsisten di berbagai eksperimen.

In [40]:
def signal_power(x):
    return np.mean(x ** 2)


def add_noise_at_snr(clean, noise, target_snr_db):
    p_signal = signal_power(clean)
    p_noise = signal_power(noise)
    target_ratio = 10 ** (target_snr_db / 10)
    scale = np.sqrt(p_signal / (target_ratio * p_noise + 1e-12))
    return clean + scale * noise


def gen_baseline_wander(n_samples, fs, freq_range=(0.15, 0.3), amplitude=1.0):
    t = np.arange(n_samples) / fs
    freq = np.random.uniform(*freq_range)
    phase = np.random.uniform(0, 2 * np.pi)
    return amplitude * np.sin(2 * np.pi * freq * t + phase)


def gen_powerline_interference(n_samples, fs, line_freq=50.0, amplitude=1.0):
    t = np.arange(n_samples) / fs
    return amplitude * np.sin(2 * np.pi * line_freq * t)


def gen_muscle_noise(n_samples, amplitude=1.0):
    return amplitude * np.random.randn(n_samples)


def apply_missing_lead(ecg, lead_indices_to_drop):
    ecg_corrupted = ecg.copy()
    ecg_corrupted[lead_indices_to_drop, :] = 0.0
    return ecg_corrupted


def augment_ecg(ecg, fs=TARGET_FS, snr_db=5.0, noise_type="baseline_wander", missing_leads=None):
    n_leads, T = ecg.shape
    noisy = np.zeros_like(ecg)
    for lead in range(n_leads):
        clean = ecg[lead]
        if noise_type == "baseline_wander":
            noise = gen_baseline_wander(T, fs)
        elif noise_type == "powerline":
            noise = gen_powerline_interference(T, fs)
        elif noise_type == "muscle":
            noise = gen_muscle_noise(T)
        elif noise_type == "clean":
            noise = np.zeros(T)
        else:
            raise ValueError(f"noise_type tidak dikenal: {noise_type}")
        noisy[lead] = add_noise_at_snr(clean, noise, target_snr_db=snr_db)
    if missing_leads:
        noisy = apply_missing_lead(noisy, missing_leads)
    return noisy


MISSING_LEAD_SCENARIOS = {
    "full": [],
    "missing_v1": [LEAD_NAMES.index("V1")],
    "missing_limb": [LEAD_NAMES.index(l) for l in ["I", "II", "III", "aVR", "aVL", "aVF"]],
    "missing_precordial": [LEAD_NAMES.index(l) for l in ["V1", "V2", "V3", "V4", "V5", "V6"]],
    "missing_random_2": None,
}


def get_missing_leads(scenario_name):
    if scenario_name == "missing_random_2":
        return sorted(random.sample(range(N_LEADS), 2))
    return MISSING_LEAD_SCENARIOS[scenario_name]

## Tahap 4: Dataset dan Pembagian Data

Pembagian data dilakukan berbasis sumber, bukan pembagian acak dari gabungan seluruh data.
PTB-XL dibagi menjadi train, validation, dan test internal. Seluruh Chapman-Shaoxing
diperlakukan sebagai test eksternal untuk mengukur generalisasi lintas dataset.

Catatan penting mengenai penanganan noise dan missing-lead: kedua jenis gangguan ini
bersifat independen satu sama lain. `noise_type="clean"` berarti tidak ada augmentasi
sama sekali, sedangkan `noise_type="no_noise"` berarti tidak ada noise tambahan tetapi
skenario missing-lead tetap diterapkan jika diminta. Pemisahan ini penting agar kondisi
missing-lead pada Eksperimen B dan C benar-benar teruji, bukan diam-diam diabaikan.

In [41]:
class ECGDataset(Dataset):
    def __init__(self, df, augment=False, snr_db=5.0, noise_type="baseline_wander",
                 missing_lead_scenario="full", cache=None):
        self.df = df.reset_index(drop=True)
        self.augment = augment
        self.snr_db = snr_db
        self.noise_type = noise_type
        self.missing_lead_scenario = missing_lead_scenario
        self.cache = cache if cache is not None else {}

    def __len__(self):
        return len(self.df)

    def _load_signal(self, record_path):
        if record_path in self.cache:
            return self.cache[record_path]
        signal = load_and_preprocess(record_path)
        self.cache[record_path] = signal
        return signal

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        signal = self._load_signal(row["record_path"]).copy()

        if self.augment:
            missing_leads = get_missing_leads(self.missing_lead_scenario)
            effective_noise_type = "clean" if self.noise_type == "no_noise" else self.noise_type
            signal = augment_ecg(
                signal, fs=TARGET_FS, snr_db=self.snr_db,
                noise_type=effective_noise_type, missing_leads=missing_leads,
            )

        x = torch.from_numpy(signal.astype(np.float32))
        y = torch.from_numpy(row["label_vec"].astype(np.float32))
        return x, y


def split_ptbxl_by_patient(df, train_frac=0.7, val_frac=0.15, seed=SEED):
    """
    Pembagian data berbasis pasien (patient-grouped split).
    Menggunakan kolom strat_fold resmi PTB-XL (PhysioNet):
      - Fold 1-8: Training (~80%)
      - Fold 9  : Validation (~10%)
      - Fold 10 : Test Internal (~10%)
    Dijamin 100% bebas dari kebocoran data pasien (zero patient leakage)
    karena pengelompokan fold resmi PhysioNet didasarkan pada patient_id.
    Jika strat_fold tidak ditemukan, fallback ke grouping unik patient_id.
    """
    if "strat_fold" in df.columns:
        train_df = df[df["strat_fold"].between(1, 8)].reset_index(drop=True)
        val_df   = df[df["strat_fold"] == 9].reset_index(drop=True)
        test_df  = df[df["strat_fold"] == 10].reset_index(drop=True)
        return train_df, val_df, test_df
    elif "patient_id" in df.columns:
        unique_pts = df["patient_id"].unique()
        rng = np.random.RandomState(seed)
        rng.shuffle(unique_pts)
        n_train = int(len(unique_pts) * train_frac)
        n_val = int(len(unique_pts) * val_frac)
        train_pts = set(unique_pts[:n_train])
        val_pts = set(unique_pts[n_train:n_train + n_val])
        test_pts = set(unique_pts[n_train + n_val:])
        return (df[df["patient_id"].isin(train_pts)].reset_index(drop=True),
                df[df["patient_id"].isin(val_pts)].reset_index(drop=True),
                df[df["patient_id"].isin(test_pts)].reset_index(drop=True))
    else:
        idx = np.arange(len(df))
        rng = np.random.RandomState(seed)
        rng.shuffle(idx)
        n_train = int(len(idx) * train_frac)
        n_val = int(len(idx) * val_frac)
        return (df.iloc[idx[:n_train]].reset_index(drop=True),
                df.iloc[idx[n_train:n_train + n_val]].reset_index(drop=True),
                df.iloc[idx[n_train + n_val:]].reset_index(drop=True))

def build_dataloader(dataset, batch_size=32, shuffle=False, num_workers=2, sampler=None):
    if sampler is not None:
        return DataLoader(dataset, batch_size=batch_size, sampler=sampler,
                           num_workers=num_workers, pin_memory=True)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                       num_workers=num_workers, pin_memory=True)

### Penanganan Ketidakseimbangan Kelas: Weighted Sampling

Distribusi kelas pada data training sangat tidak seimbang: NSR muncul pada sekitar 85 persen
rekaman, sedangkan LBBB hanya sekitar 3 persen. Dua mekanisme dipakai bersamaan untuk
menangani hal ini, dengan porsi yang saling melengkapi agar tidak saling menduplikasi efek:

1. Weighted sampling pada `DataLoader` training, yang secara efektif melakukan oversampling
   terhadap rekaman yang mengandung label langka dan undersampling terhadap rekaman yang
   hanya berlabel NSR. Bobot setiap sampel ditentukan oleh kelas paling langka yang ia miliki.
2. `pos_weight` pada loss function tetap dipakai, tetapi dengan versi yang diperlunak
   (akar kuadrat dari rasio asli) karena efek balancing sebagian besar sudah ditangani
   oleh sampler. Menggunakan kedua mekanisme dalam kekuatan penuh secara bersamaan
   berisiko membuat model terlalu agresif memprediksi kelas positif (precision anjlok).

In [42]:
def compute_sample_weights(df):
    label_matrix = np.stack(df["label_vec"].values)
    label_freq = label_matrix.sum(axis=0)
    inverse_freq = 1.0 / np.clip(label_freq, 1, None)

    sample_weights = np.zeros(len(df), dtype=np.float32)
    for i, vec in enumerate(df["label_vec"].values):
        present_classes = np.where(vec > 0)[0]
        sample_weights[i] = inverse_freq[present_classes].max()
    return sample_weights


def build_weighted_sampler(df):
    weights = compute_sample_weights(df)
    return torch.utils.data.WeightedRandomSampler(
        weights=torch.from_numpy(weights).double(),
        num_samples=len(df),
        replacement=True,
    )


def compute_pos_weight_softened(df, device, power=0.5):
    label_matrix = np.stack(df["label_vec"].values)
    pos_counts = label_matrix.sum(axis=0)
    neg_counts = len(df) - pos_counts
    raw_ratio = neg_counts / np.clip(pos_counts, 1, None)
    softened = raw_ratio ** power  # power=0.5 -> akar kuadrat, meredam rasio ekstrem
    return torch.tensor(softened, dtype=torch.float32, device=device)

In [43]:
df_ptbxl_train, df_ptbxl_val, df_ptbxl_test = split_ptbxl_by_patient(df_ptbxl)

print(f"PTB-XL train        : {len(df_ptbxl_train)}")
print(f"PTB-XL val          : {len(df_ptbxl_val)}")
print(f"PTB-XL test intern  : {len(df_ptbxl_test)}")
print(f"Chapman test ekstern: {len(df_chapman)}")

# Verifikasi empiris: Pastikan TIDAK ADA kebocoran data pasien (zero patient leakage)
if "patient_id" in df_ptbxl_train.columns:
    train_pts = set(df_ptbxl_train["patient_id"].dropna())
    val_pts   = set(df_ptbxl_val["patient_id"].dropna())
    test_pts  = set(df_ptbxl_test["patient_id"].dropna())
    train_test_overlap = len(train_pts.intersection(test_pts))
    train_val_overlap  = len(train_pts.intersection(val_pts))
    print(f"🔒 Leakage Check: Train-Test Overlapping Patients = {train_test_overlap} (Wajib 0)")
    print(f"🔒 Leakage Check: Train-Val  Overlapping Patients = {train_val_overlap} (Wajib 0)")
    assert train_test_overlap == 0, "FATAL: Terjadi patient leakage antara train dan test!"
    print("✅ VERIFIKASI LOLOS: 100% Bebas Patient Leakage!")

shared_cache = {}
train_ds = ECGDataset(df_ptbxl_train, augment=False, cache=shared_cache)
val_ds = ECGDataset(df_ptbxl_val, augment=False, cache=shared_cache)
test_internal_ds = ECGDataset(df_ptbxl_test, augment=False, cache=shared_cache)
test_external_ds = ECGDataset(df_chapman, augment=False)

train_sampler = build_weighted_sampler(df_ptbxl_train)

train_loader = build_dataloader(train_ds, batch_size=32, sampler=train_sampler)
val_loader = build_dataloader(val_ds, batch_size=32, shuffle=False)
test_internal_loader = build_dataloader(test_internal_ds, batch_size=32, shuffle=False)
test_external_loader = build_dataloader(test_external_ds, batch_size=32, shuffle=False)

print()
print("Distribusi kelas pada training set SEBELUM weighted sampling:")
label_matrix_train = np.stack(df_ptbxl_train["label_vec"].values)
for i, code_ in enumerate(CLASS_CODES):
    print(f"  {CLASS_NAMES[code_]:6s}: {int(label_matrix_train[:, i].sum())}")

PTB-XL train        : 14444
PTB-XL val          : 3095
PTB-XL test intern  : 3096
Chapman test ekstern: 9119

Distribusi kelas pada training set SEBELUM weighted sampling:
  AF    : 1062
  LBBB  : 369
  IAVB  : 547
  SB    : 445
  NSR   : 12273
  STach : 587


## Tahap 5: Arsitektur Model

Tiga model dibangun untuk kebutuhan Eksperimen A sampai C:

1. CNNBaseline: 1D-CNN konvensional yang memproses seluruh 12 lead sebagai channel input.
2. TransformerBaseline: Transformer ringan dengan patch embedding sepanjang waktu.
3. NCGAT: model utama, terdiri dari encoder CNN per lead, dua lapis graph attention
   antar-lead pada graf terhubung penuh (fully connected graph, 12 node), agregasi
   temporal, dan classifier multi-label.

Graph attention diimplementasikan langsung dengan PyTorch murni tanpa dependensi
library graph tambahan, karena jumlah node yang kecil (12 lead) membuat komputasi
attention pada graf terhubung penuh sangat ringan.

In [44]:
class GraphAttentionLayer(nn.Module):
    """
    Graph attention pada graf terhubung penuh dengan N node.
    Implementasi mengikuti formulasi GAT (Velickovic dkk, 2018), dengan skor
    e_ij didekomposisi menjadi a_src . h_i + a_dst . h_j agar komputasi efisien.
    """

    def __init__(self, in_dim, out_dim, n_heads=4, dropout=0.1, concat=True):
        super().__init__()
        self.n_heads = n_heads
        self.out_dim = out_dim
        self.concat = concat

        self.W = nn.Linear(in_dim, out_dim * n_heads, bias=False)
        self.a_src = nn.Parameter(torch.empty(n_heads, out_dim))
        self.a_dst = nn.Parameter(torch.empty(n_heads, out_dim))
        nn.init.xavier_uniform_(self.a_src)
        nn.init.xavier_uniform_(self.a_dst)

        self.leaky_relu = nn.LeakyReLU(0.2)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: [B, N, in_dim]
        B, N, _ = x.shape
        h = self.W(x).view(B, N, self.n_heads, self.out_dim)
        h = h.permute(0, 2, 1, 3)  # [B, H, N, D]

        src_scores = (h * self.a_src.view(1, self.n_heads, 1, self.out_dim)).sum(-1)  # [B, H, N]
        dst_scores = (h * self.a_dst.view(1, self.n_heads, 1, self.out_dim)).sum(-1)  # [B, H, N]

        e = src_scores.unsqueeze(-1) + dst_scores.unsqueeze(-2)  # [B, H, N, N]
        e = self.leaky_relu(e)
        alpha = torch.softmax(e, dim=-1)
        alpha = self.dropout(alpha)

        out = torch.matmul(alpha, h)  # [B, H, N, D]

        if self.concat:
            out = out.permute(0, 2, 1, 3).reshape(B, N, self.n_heads * self.out_dim)
        else:
            out = out.mean(dim=1)  # [B, N, D]
        return out, alpha

In [45]:
class PerLeadCNNEncoder(nn.Module):
    """
    Encoder CNN 1D yang diterapkan secara independen pada setiap lead.
    Seluruh lead diproses dengan bobot yang sama (shared weights) dengan
    cara menggabungkan dimensi batch dan lead sebelum masuk ke lapisan konvolusi.
    """

    def __init__(self, channels=(32, 64, 128), kernel_size=15, dropout=0.1):
        super().__init__()
        layers = []
        in_ch = 1
        for out_ch in channels:
            layers.append(nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size,
                                     stride=2, padding=kernel_size // 2))
            layers.append(nn.BatchNorm1d(out_ch))
            layers.append(nn.ReLU(inplace=True))
            layers.append(nn.Dropout(dropout))
            in_ch = out_ch
        self.conv = nn.Sequential(*layers)
        self.out_channels = channels[-1]

    def forward(self, x):
        # x: [B, N_LEADS, T]
        B, N, T = x.shape
        x = x.reshape(B * N, 1, T)
        x = self.conv(x)  # [B*N, C, T']
        C, T_out = x.shape[1], x.shape[2]
        x = x.view(B, N, C, T_out)
        return x  # [B, N_LEADS, C, T']

In [46]:
class NCGAT(nn.Module):
    """
    Model utama. Alur: per-lead CNN encoder, graph attention antar-lead
    pada setiap langkah waktu, agregasi temporal, lalu classifier multi-label.
    """

    def __init__(self, num_classes=NUM_CLASSES, cnn_channels=(32, 64, 128),
                 gat_hidden=32, gat_heads=4, dropout=0.2):
        super().__init__()
        self.encoder = PerLeadCNNEncoder(channels=cnn_channels, dropout=dropout)
        enc_dim = cnn_channels[-1]

        self.gat1 = GraphAttentionLayer(enc_dim, gat_hidden, n_heads=gat_heads,
                                         dropout=dropout, concat=True)
        self.gat2 = GraphAttentionLayer(gat_hidden * gat_heads, gat_hidden,
                                         n_heads=gat_heads, dropout=dropout, concat=False)

        self.elu = nn.ELU()
        self.classifier = nn.Sequential(
            nn.Linear(gat_hidden, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes),
        )

    def forward(self, x, return_attention=False):
        # x: [B, N_LEADS, T]
        B = x.shape[0]
        feat = self.encoder(x)  # [B, N, C, T']
        N, C, T_out = feat.shape[1], feat.shape[2], feat.shape[3]

        # proses graph attention pada setiap langkah waktu secara paralel
        feat = feat.permute(0, 3, 1, 2).reshape(B * T_out, N, C)  # [B*T', N, C]

        h1, alpha1 = self.gat1(feat)
        h1 = self.elu(h1)
        h2, alpha2 = self.gat2(h1)  # [B*T', N, gat_hidden]

        h2 = h2.view(B, T_out, N, -1).permute(0, 2, 3, 1)  # [B, N, gat_hidden, T']
        node_embed = h2.mean(dim=-1)  # agregasi temporal -> [B, N, gat_hidden]

        graph_embed = node_embed.mean(dim=1)  # readout mean over node -> [B, gat_hidden]
        logits = self.classifier(graph_embed)

        if return_attention:
            alpha2 = alpha2.view(B, T_out, alpha2.shape[1], N, N).mean(dim=1)  # rata-rata antar waktu
            return logits, alpha2
        return logits

In [47]:
class CNNBaseline(nn.Module):
    """Baseline CNN konvensional, seluruh lead diproses sebagai channel input."""

    def __init__(self, num_classes=NUM_CLASSES, channels=(64, 128, 256, 256), dropout=0.2):
        super().__init__()
        layers = []
        in_ch = N_LEADS
        for out_ch in channels:
            layers.append(nn.Conv1d(in_ch, out_ch, kernel_size=15, stride=2, padding=7))
            layers.append(nn.BatchNorm1d(out_ch))
            layers.append(nn.ReLU(inplace=True))
            layers.append(nn.Dropout(dropout))
            in_ch = out_ch
        self.conv = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Linear(channels[-1], num_classes)

    def forward(self, x):
        x = self.conv(x)
        x = self.pool(x).squeeze(-1)
        return self.classifier(x)

In [48]:
class TransformerBaseline(nn.Module):
    """Baseline Transformer ringan dengan patch embedding pada dimensi waktu."""

    def __init__(self, num_classes=NUM_CLASSES, patch_size=25, d_model=128,
                 n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()
        self.patch_embed = nn.Conv1d(N_LEADS, d_model, kernel_size=patch_size, stride=patch_size)
        self.pos_embed = nn.Parameter(torch.randn(1, TARGET_LENGTH // patch_size + 1, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 2,
            dropout=dropout, batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x):
        tokens = self.patch_embed(x).permute(0, 2, 1)  # [B, n_tokens, d_model]
        n_tokens = tokens.shape[1]
        tokens = tokens + self.pos_embed[:, :n_tokens, :]
        encoded = self.transformer(tokens)
        pooled = encoded.mean(dim=1)
        return self.classifier(pooled)

### Self-Test Arsitektur

Sebelum dipakai untuk training penuh, setiap model diuji dengan tensor acak untuk
memastikan seluruh dimensi tensor konsisten dan tidak ada kesalahan penyusunan layer.
Jalankan sel ini terlebih dahulu setiap kali arsitektur di atas diubah.

In [49]:
def self_test_models():
    batch_size = 4
    dummy_x = torch.randn(batch_size, N_LEADS, TARGET_LENGTH)

    print("Menjalankan self-test arsitektur model...")

    cnn = CNNBaseline()
    out = cnn(dummy_x)
    assert out.shape == (batch_size, NUM_CLASSES), f"CNNBaseline shape salah: {out.shape}"
    print(f"  CNNBaseline         OK, output shape {tuple(out.shape)}")

    transformer = TransformerBaseline()
    out = transformer(dummy_x)
    assert out.shape == (batch_size, NUM_CLASSES), f"TransformerBaseline shape salah: {out.shape}"
    print(f"  TransformerBaseline  OK, output shape {tuple(out.shape)}")

    ncgat = NCGAT()
    out = ncgat(dummy_x)
    assert out.shape == (batch_size, NUM_CLASSES), f"NCGAT shape salah: {out.shape}"
    print(f"  NCGAT                OK, output shape {tuple(out.shape)}")

    out, attn = ncgat(dummy_x, return_attention=True)
    expected_attn_shape = (batch_size, ncgat.gat2.n_heads, N_LEADS, N_LEADS)
    assert attn.shape == expected_attn_shape, f"attention shape salah: {attn.shape}, diharapkan {expected_attn_shape}"
    print(f"  NCGAT attention map  OK, shape {tuple(attn.shape)}")

    n_params_cnn = sum(p.numel() for p in cnn.parameters())
    n_params_transformer = sum(p.numel() for p in transformer.parameters())
    n_params_ncgat = sum(p.numel() for p in ncgat.parameters())
    print()
    print(f"Jumlah parameter CNNBaseline        : {n_params_cnn:,}")
    print(f"Jumlah parameter TransformerBaseline : {n_params_transformer:,}")
    print(f"Jumlah parameter NCGAT               : {n_params_ncgat:,}")
    print()
    print("Seluruh self-test arsitektur berhasil.")


self_test_models()

Menjalankan self-test arsitektur model...
  CNNBaseline         OK, output shape (4, 6)
  TransformerBaseline  OK, output shape (4, 6)
  NCGAT                OK, output shape (4, 6)
  NCGAT attention map  OK, shape (4, 4, 12, 12)

Jumlah parameter CNNBaseline        : 1,612,614
Jumlah parameter TransformerBaseline : 329,990
Jumlah parameter NCGAT               : 190,534

Seluruh self-test arsitektur berhasil.


## Utilitas Training dan Evaluasi

Fungsi berikut dipakai bersama oleh seluruh eksperimen: perhitungan bobot kelas untuk
menangani ketidakseimbangan data, loop training standar, loop training dengan
noise-consistency regularization, serta perhitungan metrik evaluasi.

In [52]:
def compute_metrics(y_true, y_prob, threshold=0.5):
    if np.isscalar(threshold):
        y_pred = (y_prob >= threshold).astype(int)
    else:
        y_pred = (y_prob >= threshold.reshape(1, -1)).astype(int)
    metrics = {
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
    }
    try:
        metrics["auroc_macro"] = roc_auc_score(y_true, y_prob, average="macro")
    except ValueError:
        metrics["auroc_macro"] = float("nan")
    try:
        metrics["auprc_macro"] = average_precision_score(y_true, y_prob, average="macro")
    except ValueError:
        metrics["auprc_macro"] = float("nan")
    return metrics


def find_best_thresholds(y_true, y_prob, grid=None):
    if grid is None:
        grid = np.arange(0.05, 0.96, 0.05)
    n_classes = y_true.shape[1]
    best_thresholds = np.full(n_classes, 0.5)
    for c in range(n_classes):
        best_f1, best_t = 0.0, 0.5
        for t in grid:
            f1 = f1_score(y_true[:, c], (y_prob[:, c] >= t).astype(int), zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        best_thresholds[c] = best_t
    return best_thresholds

In [53]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item() * x.size(0)
    return total_loss / len(loader.dataset)


def train_one_epoch_noise_consistency(model, loader, optimizer, criterion,
                                       device, lam=1.0, snr_db=5.0):
    model.train()
    total_loss = 0.0
    missing_scenarios = ["full", "full", "missing_v1", "missing_limb", "missing_random_2"]
    for x_clean, y in loader:
        x_clean, y = x_clean.to(device), y.to(device)

        x_noisy = x_clean.clone().cpu().numpy()
        for i in range(x_noisy.shape[0]):
            noise_type = random.choice(["baseline_wander", "muscle", "powerline"])
            scenario = random.choice(missing_scenarios)
            missing_leads = get_missing_leads(scenario)
            x_noisy[i] = augment_ecg(x_noisy[i], fs=TARGET_FS, snr_db=snr_db,
                                      noise_type=noise_type, missing_leads=missing_leads)
        x_noisy = torch.from_numpy(x_noisy).to(device)

        optimizer.zero_grad()
        logits_clean = model(x_clean)
        logits_noisy = model(x_noisy)

        loss_cls = criterion(logits_clean, y)
        prob_clean = torch.sigmoid(logits_clean)
        prob_noisy = torch.sigmoid(logits_noisy)
        loss_consistency = F.mse_loss(prob_clean, prob_noisy)

        loss = loss_cls + lam * loss_consistency
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item() * x_clean.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate_model(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []
    for x, y in loader:
        x = x.to(device)
        logits = model(x)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(y.numpy())
    y_prob = np.concatenate(all_probs, axis=0)
    y_true = np.concatenate(all_labels, axis=0)
    return compute_metrics(y_true, y_prob), y_true, y_prob

In [54]:
def run_training(model, train_loader, val_loader, device, epochs=20, lr=1e-3,
                  use_noise_consistency=False, lam=1.0, patience=5, pos_weight_power=0.5):
    model = model.to(device)
    # pos_weight diperlunak (pos_weight_power < 1) karena weighted sampler pada
    # train_loader sudah menangani sebagian besar efek ketidakseimbangan kelas.
    pos_weight = compute_pos_weight_softened(df_ptbxl_train, device, power=pos_weight_power)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_f1 = -1.0
    best_state = None
    epochs_no_improve = 0

    for epoch in range(epochs):
        if use_noise_consistency:
            train_loss = train_one_epoch_noise_consistency(
                model, train_loader, optimizer, criterion, device, lam=lam)
        else:
            train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)

        scheduler.step()
        val_metrics, _, _ = evaluate_model(model, val_loader, device)
        val_f1 = val_metrics["macro_f1"]

        print(f"Epoch {epoch+1:02d}/{epochs}  train_loss={train_loss:.4f}  "
              f"val_macro_f1={val_f1:.4f}  val_auroc={val_metrics['auroc_macro']:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping pada epoch {epoch+1}, tidak ada perbaikan {patience} epoch berturut-turut.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_val_f1

## Eksperimen A: Baseline pada Data Bersih

Tiga arsitektur dilatih dan dievaluasi pada kondisi sinyal bersih, dataset PTB-XL,
untuk mendapatkan performa dasar sebelum masuk ke eksperimen robustness.

Model NC-GAT dibangun dengan kapasitas yang dinaikkan (`gat_hidden=64`, `gat_heads=8`)
dibandingkan versi awal, agar jumlah parameternya sebanding dengan CNNBaseline dan tidak
menjadi bottleneck kapasitas. Learning rate untuk GAT juga diturunkan karena arsitektur
ini menunjukkan ketidakstabilan pada learning rate yang lebih tinggi.

Setelah training, threshold klasifikasi per kelas dikalibrasi ulang menggunakan validation
set (bukan memakai threshold tetap 0.5), karena distribusi kelas yang timpang membuat
threshold optimal berbeda-beda antar kelas.

In [55]:
model_configs = {
    "CNN": {"cls": CNNBaseline, "kwargs": {}, "epochs": 25, "lr": 1e-3, "patience": 6},
    "Transformer": {"cls": TransformerBaseline, "kwargs": {}, "epochs": 25, "lr": 5e-4, "patience": 6},
    "GAT": {"cls": NCGAT, "kwargs": {"gat_hidden": 64, "gat_heads": 8}, "epochs": 30, "lr": 5e-4, "patience": 8},
}

results_a = {}
trained_models = {}
best_thresholds_per_model = {}

for name, cfg in model_configs.items():
    print(f"=== Melatih {name} (Eksperimen A) ===")
    model = cfg["cls"](**cfg["kwargs"])
    trained_model, best_val_f1 = run_training(
        model, train_loader, val_loader, DEVICE,
        epochs=cfg["epochs"], lr=cfg["lr"], patience=cfg["patience"],
    )

    _, y_val_true, y_val_prob = evaluate_model(trained_model, val_loader, DEVICE)
    thresholds = find_best_thresholds(y_val_true, y_val_prob)
    best_thresholds_per_model[name] = thresholds

    _, y_test_true, y_test_prob = evaluate_model(trained_model, test_internal_loader, DEVICE)
    test_metrics_default = compute_metrics(y_test_true, y_test_prob, threshold=0.5)
    test_metrics_tuned = compute_metrics(y_test_true, y_test_prob, threshold=thresholds)

    print(f"{name} - test internal (threshold 0.5)   : {test_metrics_default}")
    print(f"{name} - test internal (threshold tuned)  : {test_metrics_tuned}")
    print(f"{name} - threshold per kelas: "
          f"{dict(zip([CLASS_NAMES[c] for c in CLASS_CODES], thresholds.round(2)))}")
    print()

    results_a[name] = test_metrics_tuned
    trained_models[name] = trained_model

df_results_a = pd.DataFrame(results_a).T
print(df_results_a)

=== Melatih CNN (Eksperimen A) ===
Epoch 01/25  train_loss=0.4489  val_macro_f1=0.4477  val_auroc=0.9412
Epoch 02/25  train_loss=0.2873  val_macro_f1=0.5509  val_auroc=0.9587
Epoch 03/25  train_loss=0.2412  val_macro_f1=0.4140  val_auroc=0.9622
Epoch 04/25  train_loss=0.2064  val_macro_f1=0.5952  val_auroc=0.9668
Epoch 05/25  train_loss=0.1810  val_macro_f1=0.6217  val_auroc=0.9691
Epoch 06/25  train_loss=0.1614  val_macro_f1=0.5829  val_auroc=0.9689
Epoch 07/25  train_loss=0.1435  val_macro_f1=0.5987  val_auroc=0.9690
Epoch 08/25  train_loss=0.1274  val_macro_f1=0.6396  val_auroc=0.9670
Epoch 09/25  train_loss=0.1120  val_macro_f1=0.6200  val_auroc=0.9701
Epoch 10/25  train_loss=0.1023  val_macro_f1=0.6255  val_auroc=0.9699
Epoch 11/25  train_loss=0.0911  val_macro_f1=0.6103  val_auroc=0.9710
Epoch 12/25  train_loss=0.0818  val_macro_f1=0.6652  val_auroc=0.9701
Epoch 13/25  train_loss=0.0716  val_macro_f1=0.6324  val_auroc=0.9704
Epoch 14/25  train_loss=0.0637  val_macro_f1=0.6633  va

## Eksperimen B: Robustness terhadap Noise dan Missing-Lead

Ketiga model yang sudah dilatih pada Eksperimen A dievaluasi ulang pada kondisi sinyal
yang diberi gangguan, tanpa dilatih ulang. Tujuannya mengukur seberapa besar performa
menurun ketika kondisi sinyal tidak ideal.

In [56]:
# Setiap kondisi didefinisikan sebagai (nama, skenario_missing_lead, jenis_noise, snr_db).
# noise_type "clean" berarti tidak ada augmentasi sama sekali (baseline murni).
# noise_type "no_noise" berarti tanpa noise tambahan, tetapi skenario missing-lead
# (jika bukan "full") tetap diterapkan. Pemisahan ini memperbaiki bug pada versi
# sebelumnya, di mana kondisi missing-lead murni secara tidak sengaja tidak pernah
# benar-benar diterapkan karena augmentasi ikut dinonaktifkan.
ROBUSTNESS_CONDITIONS = [
    ("clean", "full", "clean", None),
    ("baseline_wander_snr5", "full", "baseline_wander", 5.0),
    ("muscle_snr0", "full", "muscle", 0.0),
    ("missing_v1", "missing_v1", "no_noise", None),
    ("missing_limb", "missing_limb", "no_noise", None),
    ("missing_precordial", "missing_precordial", "no_noise", None),
]


def evaluate_under_condition(model, df, device, noise_type, missing_scenario, snr_db,
                              threshold=0.5):
    has_missing_lead = missing_scenario != "full"
    has_noise = noise_type not in ("clean", "no_noise")
    should_augment = has_noise or has_missing_lead

    ds = ECGDataset(
        df, augment=should_augment,
        snr_db=snr_db if snr_db is not None else 5.0,
        noise_type=noise_type, missing_lead_scenario=missing_scenario,
    )
    loader = build_dataloader(ds, batch_size=32, shuffle=False)
    _, y_true, y_prob = evaluate_model(model, loader, device)
    return compute_metrics(y_true, y_prob, threshold=threshold)


results_b = []
for model_name, model in trained_models.items():
    thresholds = best_thresholds_per_model[model_name]
    for cond_name, missing_scenario, noise_type, snr_db in ROBUSTNESS_CONDITIONS:
        metrics = evaluate_under_condition(model, df_ptbxl_test, DEVICE,
                                            noise_type=noise_type,
                                            missing_scenario=missing_scenario,
                                            snr_db=snr_db, threshold=thresholds)
        row = {"model": model_name, "condition": cond_name}
        row.update(metrics)
        results_b.append(row)
        print(f"{model_name:12s} | {cond_name:22s} | macro_f1={metrics['macro_f1']:.4f}")

df_results_b = pd.DataFrame(results_b)

CNN          | clean                  | macro_f1=0.7570
CNN          | baseline_wander_snr5   | macro_f1=0.5588
CNN          | muscle_snr0            | macro_f1=0.6243
CNN          | missing_v1             | macro_f1=0.7089
CNN          | missing_limb           | macro_f1=0.5448
CNN          | missing_precordial     | macro_f1=0.3687
Transformer  | clean                  | macro_f1=0.6076
Transformer  | baseline_wander_snr5   | macro_f1=0.4987
Transformer  | muscle_snr0            | macro_f1=0.4433
Transformer  | missing_v1             | macro_f1=0.5510
Transformer  | missing_limb           | macro_f1=0.2679
Transformer  | missing_precordial     | macro_f1=0.1029
GAT          | clean                  | macro_f1=0.7153
GAT          | baseline_wander_snr5   | macro_f1=0.6881
GAT          | muscle_snr0            | macro_f1=0.0160
GAT          | missing_v1             | macro_f1=0.3377
GAT          | missing_limb           | macro_f1=0.2403
GAT          | missing_precordial     | macro_f1

## Eksperimen C: Noise-Consistency Training

Model GAT dilatih ulang dari awal dengan noise-consistency regularization, kemudian
dibandingkan dengan model GAT dari Eksperimen A yang dilatih tanpa regularization
tersebut, pada kondisi sinyal yang sama dengan Eksperimen B.

In [57]:
print("=== Melatih GAT dengan Noise-Consistency Training (Eksperimen C) ===")
model_gat_nc = NCGAT(gat_hidden=64, gat_heads=8)
trained_gat_nc, best_val_f1_nc = run_training(
    model_gat_nc, train_loader, val_loader, DEVICE, epochs=30, lr=5e-4, patience=8,
    use_noise_consistency=True, lam=1.0,
)

_, y_val_true_nc, y_val_prob_nc = evaluate_model(trained_gat_nc, val_loader, DEVICE)
thresholds_gat_nc = find_best_thresholds(y_val_true_nc, y_val_prob_nc)
thresholds_gat_plain = best_thresholds_per_model["GAT"]

results_c = []
for cond_name, missing_scenario, noise_type, snr_db in ROBUSTNESS_CONDITIONS:
    metrics_no_nc = evaluate_under_condition(trained_models["GAT"], df_ptbxl_test, DEVICE,
                                              noise_type=noise_type,
                                              missing_scenario=missing_scenario, snr_db=snr_db,
                                              threshold=thresholds_gat_plain)
    metrics_with_nc = evaluate_under_condition(trained_gat_nc, df_ptbxl_test, DEVICE,
                                                noise_type=noise_type,
                                                missing_scenario=missing_scenario, snr_db=snr_db,
                                                threshold=thresholds_gat_nc)
    results_c.append({
        "condition": cond_name,
        "macro_f1_without_nc": metrics_no_nc["macro_f1"],
        "macro_f1_with_nc": metrics_with_nc["macro_f1"],
        "delta": metrics_with_nc["macro_f1"] - metrics_no_nc["macro_f1"],
    })

df_results_c = pd.DataFrame(results_c)
print(df_results_c)
print()
print("Nilai delta positif menunjukkan noise-consistency training membantu pada kondisi tersebut.")

=== Melatih GAT dengan Noise-Consistency Training (Eksperimen C) ===
Epoch 01/30  train_loss=0.7328  val_macro_f1=0.2611  val_auroc=0.7853
Epoch 02/30  train_loss=0.5895  val_macro_f1=0.2692  val_auroc=0.8108
Epoch 03/30  train_loss=0.5428  val_macro_f1=0.2696  val_auroc=0.8303
Epoch 04/30  train_loss=0.5143  val_macro_f1=0.4057  val_auroc=0.8636
Epoch 05/30  train_loss=0.4635  val_macro_f1=0.4533  val_auroc=0.9221
Epoch 06/30  train_loss=0.3969  val_macro_f1=0.5090  val_auroc=0.9301
Epoch 07/30  train_loss=0.3631  val_macro_f1=0.5057  val_auroc=0.9406
Epoch 08/30  train_loss=0.3384  val_macro_f1=0.4846  val_auroc=0.9447
Epoch 09/30  train_loss=0.3187  val_macro_f1=0.5271  val_auroc=0.9451
Epoch 10/30  train_loss=0.3073  val_macro_f1=0.4575  val_auroc=0.9501
Epoch 11/30  train_loss=0.2825  val_macro_f1=0.5177  val_auroc=0.9529
Epoch 12/30  train_loss=0.2825  val_macro_f1=0.5175  val_auroc=0.9555
Epoch 13/30  train_loss=0.2648  val_macro_f1=0.5101  val_auroc=0.9501
Epoch 14/30  train_lo

## Eksperimen D: Evaluasi Cross-Dataset

Model GAT dengan noise-consistency training dievaluasi pada dataset eksternal
Chapman-Shaoxing untuk mengukur generalisasi lintas dataset (domain shift), dan
dibandingkan dengan performanya pada dataset sumber PTB-XL.

In [58]:
_, y_source_true, y_source_prob = evaluate_model(trained_gat_nc, test_internal_loader, DEVICE)
_, y_target_true, y_target_prob = evaluate_model(trained_gat_nc, test_external_loader, DEVICE)

metrics_source = compute_metrics(y_source_true, y_source_prob, threshold=thresholds_gat_nc)
metrics_target = compute_metrics(y_target_true, y_target_prob, threshold=thresholds_gat_nc)

delta_f1 = metrics_source["macro_f1"] - metrics_target["macro_f1"]

print("Performa pada PTB-XL (source):", metrics_source)
print("Performa pada Chapman-Shaoxing (target):", metrics_target)
print(f"Delta F1 (source - target): {delta_f1:.4f}")
print("Semakin kecil delta F1, semakin baik generalisasi model lintas dataset.")

df_results_d = pd.DataFrame([
    {"dataset": "PTB-XL (source)", **metrics_source},
    {"dataset": "Chapman-Shaoxing (target)", **metrics_target},
])

Performa pada PTB-XL (source): {'macro_f1': 0.6968272736839909, 'micro_f1': 0.8796524257784214, 'auroc_macro': np.float64(0.9642319361979013), 'auprc_macro': np.float64(0.7544593498425973)}
Performa pada Chapman-Shaoxing (target): {'macro_f1': 0.6153980413738386, 'micro_f1': 0.5935296764838242, 'auroc_macro': np.float64(0.9274416949086645), 'auprc_macro': np.float64(0.7290671677402031)}
Delta F1 (source - target): 0.0814
Semakin kecil delta F1, semakin baik generalisasi model lintas dataset.


## Eksperimen E: Kestabilan Explainable AI

Kepentingan setiap lead (lead importance) dihitung menggunakan metode saliency berbasis
gradien, kemudian dibandingkan pada dua kondisi: sinyal bersih dibandingkan sinyal
dengan noise, dan dataset sumber dibandingkan dataset target. Empat metrik kestabilan
dipakai untuk kuantifikasi konsistensi: korelasi Spearman, cosine similarity, top-k
lead overlap, dan Jensen-Shannon divergence.

In [59]:
def compute_lead_importance(model, x, device):
    model.eval()
    x = x.clone().to(device)
    x.requires_grad_(True)
    logits = model(x)
    score = logits.sum()
    score.backward()
    grad = x.grad.detach().cpu().numpy()  # [B, N_LEADS, T]
    importance = np.abs(grad).sum(axis=-1)  # [B, N_LEADS]
    importance = importance / (importance.sum(axis=-1, keepdims=True) + 1e-12)
    return importance


def stability_metrics(importance_a, importance_b, top_k=3):
    spearman_corr, _ = spearmanr(importance_a, importance_b)
    cosine_sim = cosine_similarity(importance_a.reshape(1, -1), importance_b.reshape(1, -1))[0, 0]

    top_a = set(np.argsort(importance_a)[-top_k:])
    top_b = set(np.argsort(importance_b)[-top_k:])
    overlap = len(top_a & top_b) / top_k

    eps = 1e-12
    p = importance_a + eps
    q = importance_b + eps
    p = p / p.sum()
    q = q / q.sum()
    js_div = jensenshannon(p, q) ** 2

    return {
        "spearman": spearman_corr,
        "cosine_similarity": cosine_sim,
        "top_k_overlap": overlap,
        "js_divergence": js_div,
    }

In [60]:
sample_batch_x, _ = next(iter(test_internal_loader))
sample_batch_x = sample_batch_x[:8]

noisy_batch = sample_batch_x.clone().numpy()
for i in range(noisy_batch.shape[0]):
    noisy_batch[i] = augment_ecg(noisy_batch[i], fs=TARGET_FS, snr_db=5.0, noise_type="baseline_wander")
noisy_batch = torch.from_numpy(noisy_batch)

importance_clean = compute_lead_importance(trained_gat_nc, sample_batch_x, DEVICE)
importance_noisy = compute_lead_importance(trained_gat_nc, noisy_batch, DEVICE)

stability_clean_vs_noisy = []
for i in range(sample_batch_x.shape[0]):
    stability_clean_vs_noisy.append(
        stability_metrics(importance_clean[i], importance_noisy[i])
    )
df_stability_noise = pd.DataFrame(stability_clean_vs_noisy)
print("Kestabilan XAI: clean vs noisy (rata-rata dari sampel)")
print(df_stability_noise.mean())

Kestabilan XAI: clean vs noisy (rata-rata dari sampel)
spearman             0.868881
cosine_similarity    0.979832
top_k_overlap        0.791667
js_divergence        0.006726
dtype: float64


In [61]:
sample_batch_chapman, _ = next(iter(test_external_loader))
sample_batch_chapman = sample_batch_chapman[:8]

importance_source = compute_lead_importance(trained_gat_nc, sample_batch_x, DEVICE)
importance_target = compute_lead_importance(trained_gat_nc, sample_batch_chapman, DEVICE)

n_compare = min(len(importance_source), len(importance_target))
stability_source_vs_target = []
for i in range(n_compare):
    stability_source_vs_target.append(
        stability_metrics(importance_source[i], importance_target[i])
    )
df_stability_domain = pd.DataFrame(stability_source_vs_target)
print("Kestabilan XAI: source (PTB-XL) vs target (Chapman-Shaoxing)")
print(df_stability_domain.mean())

Kestabilan XAI: source (PTB-XL) vs target (Chapman-Shaoxing)
spearman             0.566434
cosine_similarity    0.890667
top_k_overlap        0.583333
js_divergence        0.035905
dtype: float64


## Ringkasan Hasil Akhir

Seluruh hasil dari Eksperimen A sampai E dikumpulkan menjadi satu ringkasan dan
disimpan ke berkas CSV agar dapat dilampirkan pada laporan.

In [62]:
summary = {
    "eksperimen_a_baseline": df_results_a.to_dict(),
    "eksperimen_b_robustness": df_results_b.to_dict(orient="records"),
    "eksperimen_c_noise_consistency": df_results_c.to_dict(orient="records"),
    "eksperimen_d_cross_dataset": df_results_d.to_dict(orient="records"),
    "eksperimen_e_xai_clean_vs_noisy": df_stability_noise.mean().to_dict(),
    "eksperimen_e_xai_source_vs_target": df_stability_domain.mean().to_dict(),
}

with open("/kaggle/working/ncgat_results_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

df_results_a.to_csv("/kaggle/working/eksperimen_a_baseline.csv")
df_results_b.to_csv("/kaggle/working/eksperimen_b_robustness.csv", index=False)
df_results_c.to_csv("/kaggle/working/eksperimen_c_noise_consistency.csv", index=False)
df_results_d.to_csv("/kaggle/working/eksperimen_d_cross_dataset.csv", index=False)

print("Ringkasan hasil disimpan di /kaggle/working/")
print()
print("Eksperimen A - Baseline (test internal PTB-XL):")
print(df_results_a)
print()
print("Eksperimen C - Noise-Consistency (ringkasan delta macro F1):")
print(df_results_c[["condition", "delta"]])
print()
print("Eksperimen D - Cross-Dataset:")
print(df_results_d)

Ringkasan hasil disimpan di /kaggle/working/

Eksperimen A - Baseline (test internal PTB-XL):
             macro_f1  micro_f1  auroc_macro  auprc_macro
CNN          0.757032  0.921123     0.971542     0.791547
Transformer  0.607577  0.842575     0.895844     0.627926
GAT          0.715290  0.793388     0.971270     0.810375

Eksperimen C - Noise-Consistency (ringkasan delta macro F1):
              condition     delta
0                 clean -0.018463
1  baseline_wander_snr5 -0.005985
2           muscle_snr0  0.289822
3            missing_v1  0.353048
4          missing_limb  0.409558
5    missing_precordial  0.448004

Eksperimen D - Cross-Dataset:
                     dataset  macro_f1  micro_f1  auroc_macro  auprc_macro
0            PTB-XL (source)  0.696827  0.879652     0.964232     0.754459
1  Chapman-Shaoxing (target)  0.615398  0.593530     0.927442     0.729067
